🧩 1. Imports

In [4]:
import json
import os
from pathlib import Path
import pandas as pd
from collections import Counter
from sklearn.metrics import accuracy_score, f1_score
from itertools import combinations
from sentence_transformers import SentenceTransformer
import numpy as np

In [5]:
def load_jsonl(path):
    data = []
    with open(path, "r") as f:
        for line in f:
            data.append(json.loads(line))
    return data

In [6]:
def compute_classification_metrics(data):
    y_true = [x["gold_label"] for x in data]
    y_pred = [x["prediction"] for x in data]
    
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average="macro")
    
    return {
        "accuracy": acc,
        "macro_f1": f1
    }

In [7]:
def compute_steps_stats(data):
    steps = [len(x["pipeline"]["steps"]) for x in data]
    
    return {
        "avg_steps": np.mean(steps),
        "max_steps": np.max(steps),
        "min_steps": np.min(steps)
    }

In [8]:
def compute_evidence_stats(data):
    evidence_counts = []
    rerank_scores = []
    
    for x in data:
        for step in x["pipeline"]["steps"]:
            evs = step.get("evidence", [])
            evidence_counts.append(len(evs))
            
            for e in evs:
                if "rerank_score" in e:
                    rerank_scores.append(e["rerank_score"])
    
    return {
        "avg_evidence_per_step": np.mean(evidence_counts),
        "avg_rerank_score": np.mean(rerank_scores)
    }

In [9]:
model = SentenceTransformer("all-MiniLM-L6-v2")

def compute_query_diversity(data):
    diversities = []
    
    for x in data:
        questions = [s["question"] for s in x["pipeline"]["steps"]]
        
        if len(questions) < 2:
            continue
        
        embeddings = model.encode(questions)
        
        sims = []
        for i, j in combinations(range(len(questions)), 2):
            sim = np.dot(embeddings[i], embeddings[j]) / (
                np.linalg.norm(embeddings[i]) * np.linalg.norm(embeddings[j])
            )
            sims.append(sim)
        
        diversity = 1 - np.mean(sims)
        diversities.append(diversity)
    
    return {
        "avg_query_diversity": np.mean(diversities)
    }

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 690.96it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/home/systemp2w/miniconda3/lib/python3.13/site-packages/torch/cuda/__init__.py:435: UserWarning: 
    Found GPU1 NVIDIA GeForce GTX 1050 Ti which is of cuda capability 6.1.
    Minimum and Maximum cuda capability supported by this version of PyTorch is
    (7.5) - (12.0)
    
  queued_call()
/home/systemp2w/miniconda3/lib/python3.13/site-packages/torch/cuda/__init__.py:435: UserWarning: 
    Please install PyTorch with a following CUDA
    configurations:  12.6 following instructions at
    https://pytorch.org/get-started/locally/
    
  q

In [10]:
def compare_runs(data_a, data_b):
    results = []
    
    for a, b in zip(data_a, data_b):
        results.append({
            "claim_id": a["claim_id"],
            "gold": a["gold_label"],
            "pred_a": a["prediction"],
            "pred_b": b["prediction"],
            "a_correct": a["prediction"] == a["gold_label"],
            "b_correct": b["prediction"] == b["gold_label"]
        })
    
    df = pd.DataFrame(results)
    
    return {
        "A_better": len(df[(df.a_correct == True) & (df.b_correct == False)]),
        "B_better": len(df[(df.a_correct == False) & (df.b_correct == True)]),
        "both_correct": len(df[(df.a_correct == True) & (df.b_correct == True)]),
        "both_wrong": len(df[(df.a_correct == False) & (df.b_correct == False)])
    }

In [ ]:
base_path = "outputs"

orig_path = Path(base_path) / "averitec_original_pipeline" / "2026-03-30_01-25-13" / "averitec_dev.jsonl"
iter_path = Path(base_path) / "averitec_iterative_pipeline" / "2026-03-30_01-27-11" / "averitec_dev.jsonl"

orig = load_jsonl(orig_path)
iterative = load_jsonl(iter_path)

print("=== ORIGINAL ===")
print(compute_classification_metrics(orig))
print(compute_steps_stats(orig))
print(compute_evidence_stats(orig))

print("\n=== ITERATIVE ===")
print(compute_classification_metrics(iterative))
print(compute_steps_stats(iterative))
print(compute_evidence_stats(iterative))

print("\n=== DIVERSITY ===")
print("Original:", compute_query_diversity(orig))
print("Iterative:", compute_query_diversity(iterative))

print("\n=== COMPARISON ===")
print(compare_runs(orig, iterative))

FileNotFoundError: [Errno 2] No such file or directory: 'outputs/averitec_original_pipeline/2026-03-30_01-27-11/averitec_dev.jsonl'